# PySATL-CPD Online Algorithms Walkthrough

In [ ]:
## 1. Setup and Imports
import numpy as np
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import norm
from plotly.offline import plot

from pysatl_cpd.analysis.visualization.components import (
    VerticalLineVisualComponent,
    VerticalFillComponent
)
from pysatl_cpd.analysis.visualization.typedefs import DrawBackend
from pysatl_cpd.analysis.labeled_data import LabeledData

from typing import Any


import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

from pysatl_cpd.analysis.labeled_data import LabeledData
from pysatl_cpd.core.typedefs import UnivariateNumericArray as NumericArray
from pysatl_cpd.core.online.online_cpd_solver import OnlineCpdSolver
from pysatl_cpd.algorithms.online.shewhart_control_chart import ShewhartControlChart, ShewhartControlChartState
from pysatl_cpd.analysis.visualization.typedefs import DrawBackend
from pysatl_cpd.analysis.visualization.online.online_trace_visualizer import OnlineTraceVisualizer
from pysatl_cpd.analysis.visualization.timeseries.univariate_timeseries_visualizer import UnivariateTimeseriesVisualizer
from pysatl_cpd.analysis.visualization.online.states.state_dummy_visualizer import DummyStateVisualizer
from pysatl_cpd.analysis.visualization.online.states.state_shewhart_chart_visualizer import ShewhartStateVisualizer

from pysatl_cpd.core.online.online_detection_trace import OnlineDetectionTrace


## Prelude. Meet Data Providers

TODO

## Exposition. So what is an algorithm?

TODO

## Plot. Meet The Solver and Traces

TODO

## Unfolding of The Plot. Visualisation

### Preparing data and running algorithm

We first will generate some sample data with change-points from normal distribution and visualise it without manually

In [ ]:
import matplotlib.pyplot as plt 

def generate_labeled_data(
    data_len: int,
    means: list[float],
    change_point_index: list[int],
    var: float = 1.0
) -> LabeledData[np.float64]:
    """
    Generate synthetic labeled time series data with change points.

    Parameters
    ----------
    means : list[float]
        Mean values for each segment.
    lengths : list[int]
        Lengths of each segment.
    var : float, default=1.0
        Variance of the normal distribution.

    Returns
    -------
    LabeledData[np.float64]
        Labeled dataset with generated observations and change point indices.
    """
    idxs = [0] + change_point_index + [data_len]
    lengths = [idxs[i+1] - idxs[i] for i in range(len(change_point_index) + 1)]
    if len(means) != len(lengths):
        raise ValueError("Length of means and lengths mismatch")

    raw_data: NumericArray = np.empty(shape=(0,))
    change_points: list[int] = [0]
    for mean, length in zip(means, lengths, strict=True):
        raw_data = np.concatenate(
            (
                raw_data,
                np.array(
                    norm.rvs(size=length, loc=mean, scale=np.sqrt(var))
                ).reshape((-1,))
            ),
            axis=0
        )
        change_points.append(change_points[-1] + length)
    return LabeledData(raw_data=raw_data, change_points=change_points[1:-1])


data_len = 12_000
change_point_index = [1_000, 7_500, 10_500]
change_point_means = [0, 4, -2, 2]
data = generate_labeled_data(data_len, change_point_means, change_point_index)

idxs = [0] + change_point_index + [data_len]
mean = [change_point_means[i] for i in range(len(change_point_index) + 1)  for _ in range(idxs[i+1] - idxs[i])]



#Create figure with 3 subplots
fig, ax1 = plt.subplots(1, 1, figsize=(14, 4), sharex=True)
fig.suptitle("Manual Visualization of Online Change-Point Detection", fontsize=16, fontweight='bold')

ax1.plot(np.arange(len(data)), np.array(list(data)), 'k-', linewidth=1, alpha=0.7, label='Time Series')
margin = 200

for i, cp in enumerate(data.change_points):
    # Add ground truth change points
    ax1.axvline(x=cp, color='red', linestyle='--', linewidth=2, alpha=0.8, 
               label='Ground Truth' if i == 0 else '')
    # Add ground truth margins (window around truth)
    ax1.axvspan(cp - margin, cp + margin, alpha=0.1, color='red',
               label='Margin Window' if i == 0 else '')

ax1.set_ylabel('Value')
ax1.set_title('Time Series with Change Points')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Adjust layout
plt.tight_layout()
plt.show()


Now let's run algorithm on our data

In [ ]:
from dataclasses import fields
from pysatl_cpd.algorithms.online.shewhart_control_chart import ShewhartControlChart
from pysatl_cpd.core.online import OnlineCpdSolver, OnlineDetectionTrace

# Create Shewhart control chart algorithm
learning_period_size = 100
window_size = 50
algorithm = ShewhartControlChart(
    learning_period_size=100,
    window_size=50
)

# Run online detection solver
solver = OnlineCpdSolver(
    skip_period=40,
    max_runlength=1000,
    collect_states=True  # Enable state collection for learning periods
)

# Collect detection steps
steps = list(solver.run(algorithm, data, 2.5))

# Build detection trace from steps
trace = OnlineDetectionTrace.from_run(
    algorithm_name = algorithm.name,
    configuration_hash = str(hash(algorithm.configuration)),
    threshold=2.5,
    steps=steps
)

print('Trace contents:\n├── ' + '\n├── '.join(f"{field.name}: {field.type}"  for field in fields(trace)))

Let's visualize trace content. For now we will go by hand and first we should extract information from trace into nice form

We now ready to visualize our trace

In [ ]:
#Create figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Manual Visualization of Online Change-Point Detection", fontsize=16, fontweight='bold')

# Get data
time_points = np.arange(len(data))
values = np.array(list(data))

ax1, ax2, ax3 = axes[0], axes[1], axes[2]

# Subplot 1: Time series with change points
ax1.plot(time_points, values, 'k-', linewidth=1, alpha=0.7, label='Time Series')

for i, cp in enumerate(data.change_points):
    # Add ground truth change points
    ax1.axvline(x=cp, color='red', linestyle='-', linewidth=2, alpha=0.8, 
               label='Ground Truth' if i == 0 else '')
    # Add ground truth margins (window around truth)
    ax1.axvspan(cp - margin, cp + margin, alpha=0.1, color='red',
               label='Margin Window' if i == 0 else '')

## Add detected change points
for i, cp in enumerate(trace.signal_change_points):
    ax1.axvline(x=cp, color='green', linestyle='--', linewidth=2, alpha=0.8,
               label='Detected CP' if i == 0 else '')

## Add forced change points
for i, cp in enumerate(trace.forced_change_points):
    ax1.axvline(x=cp, color='orange', linestyle='--', linewidth=2, alpha=0.8,
               label='Forced CP' if i == 0 else '')

ax1.set_ylabel('Value')
ax1.set_title('Time Series with Change Points')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)


# Subplot 2: Detection function with threshold
detection_scores = trace.detection_function
time_scores = np.arange(len(detection_scores))

ax2.plot(time_scores, detection_scores, 'b-', linewidth=1, alpha=0.7, label='Detection Function')
ax2.axhline(y=2.5, color='red', linestyle='--', linewidth=2, alpha=0.8, label='Threshold (2.5)')
ax2.set_ylabel('Detection Statistic')
ax2.set_title('Detection Function')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)


# Subplot 3: Processing time
processing_times = trace.processing_time
time_proc = np.arange(len(processing_times))

ax3.plot(time_proc, processing_times, 'purple', linewidth=1, alpha=0.7, label='Processing Time')
ax3.fill_between(time_proc, 0, processing_times, alpha=0.3, color='purple')

ax3.set_xlabel('Time Index')
ax3.set_ylabel('Time (seconds)')
ax3.set_title('Processing Time per Step')
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

# Commons: skip and learning periods

## Add skip periods (visualize as shaded regions)
for i, (s, e) in enumerate(trace.skip_periods):
    ax1.axvspan(s, e, alpha=0.2, color='brown', 
               label='Skip Period' if i == 0 else '')
    ax2.axvspan(s, e, alpha=0.2, color='brown', 
               label='Skip Period' if i == 0 else '')
    ax3.axvspan(s, e, alpha=0.2, color='brown', 
               label='Skip Period' if i == 0 else '')

## Add learning periods (visualize as shaded regions)
for i, (s, e) in enumerate(trace.learning_periods):
    ax1.axvspan(s, e, alpha=0.2, color='green', 
               label='Learning Period' if i == 0 else '')
    ax2.axvspan(s, e, alpha=0.2, color='green', 
               label='Learning Period' if i == 0 else '')
    ax3.axvspan(s, e, alpha=0.2, color='green', 
               label='Learning Period' if i == 0 else '')
  
# Adjust layout and show image
plt.tight_layout()
plt.show()

We now should see how to use visualisation utills from PySATL-CPD module

### Level I: Visual components

Visual components are follow TODO interface which can be described as follows:
```python
TODO
```
They are usefull for TODO. For now, there are two main visual components:
1. Vertical line (used for TODO)
2. Vertical fill (used for TODO)

In this section we are going to show, how use those components, to simplify plot creation.

#### Creating components

First we will define visual components: we will use them across this tutorial

In [ ]:
# Ground truth lines component
ground_truth_lines = (
    VerticalLineVisualComponent(DrawBackend.MATPLOTLIB)
    .set_style(color="red", linestyle="solid", linewidth=2, alpha=0.8)
    .set_lines(data.change_points)
    .set_legend_label("Ground Truth")
)

# Ground truth margin fill component
margin_fill = (
    VerticalFillComponent(DrawBackend.MATPLOTLIB)
    .set_style(fill_color="red", fill_alpha=0.1)
    .set_regions([(cp - margin, cp + margin) for cp in data.change_points])
    .set_legend_label("Margin Window")
)

# Detected lines component (excluding forced)
detected_lines = (
    VerticalLineVisualComponent(DrawBackend.MATPLOTLIB)
    .set_style(color="green", linestyle="dash", linewidth=2, alpha=0.8)
    .set_lines(trace.signal_change_points)
    .set_legend_label("Detected CP")
)

# Forced lines component
forced_lines = (
    VerticalLineVisualComponent(DrawBackend.MATPLOTLIB)
    .set_style(color="orange", linestyle="dash", linewidth=2, alpha=0.8)
    .set_lines(trace.forced_change_points)
    .set_legend_label("Forced CP")
)

# Skip periods fill component
skip_fill = (
    VerticalFillComponent(DrawBackend.MATPLOTLIB)
    .set_style(fill_color="brown", fill_alpha=0.2)
    .set_regions(trace.skip_periods)
    .set_legend_label("Skip Period")
)

# Learning periods fill component
learning_fill = (
    VerticalFillComponent(DrawBackend.MATPLOTLIB)
    .set_style(fill_color="green", fill_alpha=0.2)
    .set_regions(trace.learning_periods)
    .set_legend_label("Learning Period")
)


alg_states_components = [learning_fill, skip_fill]
detection_components = [ground_truth_lines, margin_fill, detected_lines, forced_lines]

#### Example: drawing trace with visual components (MatPlotLib)

In [ ]:
# Create figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Visualization of Online Change-Point Detection using visual components", fontsize=16, fontweight='bold')

ax1, ax2, ax3 = axes[0], axes[1], axes[2]

# Get data
time_points = np.arange(len(data))
values = np.array(list(data))

# ========== Subplot 1: Time series ==========
# Draw time series line
ax1.plot(time_points, values, color="black", linewidth=1, alpha=0.7, label="Time Series")
ax1.set_ylabel("Value")
ax1.set_title("Time Series with Change Points")
ax1.grid(True, alpha=0.3)

# ========== Subplot 2: Detection function ==========

# Draw detection function line
detection_scores = trace.detection_function
time_scores = np.arange(len(detection_scores))
ax2.plot(time_scores, detection_scores, color="blue", linewidth=1, alpha=0.7, label="Detection Function")
ax2.axhline(y=2.5, color="red", linestyle="--", linewidth=2, alpha=0.8, label="Threshold (2.5)")
ax2.set_ylabel("Detection Statistic")
ax2.set_title("Detection Function")
ax2.grid(True, alpha=0.3)

# ========== Subplot 3: Processing time ==========
# Draw processing time line and fill
processing_times = trace.processing_time
time_proc = np.arange(len(processing_times))
ax3.plot(time_proc, processing_times, color="purple", linewidth=1, alpha=0.7, label="Processing Time")
ax3.fill_between(time_proc, 0, processing_times, alpha=0.3, color="purple")

ax3.set_xlabel("Time Index")
ax3.set_ylabel("Time (seconds)")
ax3.set_title("Processing Time per Step")
ax3.grid(True, alpha=0.3)

# ========== Drawing components and legens ==========
# Draw fills and lines
for component in detection_components:
    component.draw(fig, ax1, add_legend = True)
ax1.legend(loc="upper left", fontsize=9)

for component in alg_states_components:
    component.draw(fig, ax1, add_legend = False)
    component.draw(fig, ax3, add_legend = False)
    component.draw(fig, ax2, add_legend = True)

for ax in [ax1, ax2, ax3]:
    ax.legend(loc="upper left", fontsize=9)

# Adjust layout
plt.tight_layout()
plt.show()

#### Example: drawing trace with visual components (Plotly)

In [ ]:
# Create figure with 3 subplots
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Time Series with Change Points",
        "Detection Function",
        "Processing Time per Step"
    )
)

# ========== Get data ==========
time_points = np.arange(len(data))
values = np.array(list(data))

# ========== Subplot 1: Time series ==========
# Draw time series line
fig.add_trace(
    go.Scatter(
        x=time_points,
        y=values,
        mode="lines",
        name="Time Series",
        line=dict(color="black", width=1)
    ),
    row=1, col=1
)

# ========== Subplot 2: Detection function ==========
detection_scores = trace.detection_function
time_scores = np.arange(len(detection_scores))

# Draw detection function line
fig.add_trace(
    go.Scatter(
        x=time_scores,
        y=detection_scores,
        mode="lines",
        name="Detection Function",
        line=dict(color="blue", width=1)
    ),
    row=2, col=1
)

# Draw threshold line (horizontal)
fig.add_hline(
    y=2.5,
    line_color="red",
    line_dash="dash",
    line_width=2,
    opacity=0.8,
    row=2, col=1,
    name="Threshold",
    showlegend=True
)


# ========== Subplot 3: Processing time ==========
processing_times = trace.processing_time
time_proc = np.arange(len(processing_times))

# Draw processing time line with fill
fig.add_trace(
    go.Scatter(
        x=time_proc,
        y=processing_times,
        mode="lines",
        name="Processing Time",
        line=dict(color="purple", width=1),
        fill="tozeroy",
        fillcolor="rgba(128, 0, 128, 0.3)"
    ),
    row=3, col=1
)

# ========== Draw components ==========
# Detection components (ground truth lines, margin fill, detected lines, forced lines)
# These go on time series subplot

# Change backend of components to PLOTLY
for component in detection_components + alg_states_components:
    component.backend = 'plotly'

# Algorithm state components (skip periods, learning periods)
# Draw on all subplots
for component in alg_states_components:
    # Draw on time series subplot (row 1)
    component.draw(fig, (1, 1), add_legend=False)
    # Draw on detection function subplot (row 2)
    component.draw(fig, (2, 1), add_legend=True)
    # Draw on processing time subplot (row 3)
    component.draw(fig, (3, 1), add_legend=False)
  
for component in detection_components:
    component.draw(fig, (1, 1), add_legend=True)
# ========== Configure layout ==========
fig.update_layout(
    title="Visualization of Online Change-Point Detection using visual components",
    title_font_size=16,
    height=900,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    hovermode="closest"
)

# Update axes labels and grid
fig.update_xaxes(title_text="Time Index", row=3, col=1)
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_yaxes(title_text="Detection Statistic", row=2, col=1)
fig.update_yaxes(title_text="Time (seconds)", row=3, col=1)

# Add grid to all subplots
for i in range(1, 4):
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="lightgray", row=i, col=1)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="lightgray", row=i, col=1)



fig.update_layout(
    legend={
        "x": 1.02,  # Position to the right of the plot
        "y": 0.95,     # Top-aligned
        "xanchor": "left",  # Anchor point at left of legend box
        "yanchor": "top",   # Anchor point at top of legend box
        "bgcolor": "rgba(255, 255, 255, 0.8)",  # Semi-transparent background
        "bordercolor": "black",
        "borderwidth": 1
    }
)

fig.show()

### Level II: Visualisers

#### Example: Using Timeseries and Trace Visualisers (MatPlotLib)

In [ ]:
# Create visual components with custom styles matching the manual example
backend = DrawBackend.MATPLOTLIB

# Create time series visualizer
timeseries_visualizer = UnivariateTimeseriesVisualizer(backend=backend)
timeseries_visualizer.set_data_provider(data)

(timeseries_visualizer
  .set_plot_opts(
    title="Time Series with Change Points",
    xlabel="Time Index",
    ylabel="Value",
    grid=True,
    legend=True
  ).set_draw_opts(
    color="black",
    linewidth=1.5,
    alpha=0.7
  )
)

# Create trace visualizer
trace_visualizer = OnlineTraceVisualizer(
    backend=backend,
    state_visualizer=DummyStateVisualizer[ShewhartControlChartState](backend=backend),
)
trace_visualizer.set_trace(trace)

# Configure detection function plot
(trace_visualizer
    .set_detection_func_plot_opts(
        title="Detection Function",
        xlabel="Time Index",
        ylabel="Detection Statistic",
        grid=True,
        legend=True
    )
    .set_detection_func_draw_opts(
        color="blue",
        linewidth=1
    )
    .set_threshold_draw_opts(
        color="red",
        linestyle="dash",
        linewidth=2,
        alpha=0.8
    )
    .set_processing_time_plot_opts(
        title="Processing Time per Step",
        xlabel="Time Index",
        ylabel="Time (seconds)",
        grid=True,
        legend=True
    )
    .set_processing_time_draw_opts(
        color="purple",
        linewidth=1,
        fill_alpha=0.3
    )
)

# Create figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Online Change-Point Detection with Shewhart Control Chart", fontsize=16, fontweight='bold')

# Map subplots to visualizers
ax_mapping = {
    "timeseries": axes[0],                    # Time series
    "detection_function": axes[1],      # Detection function
    "processing_time": axes[2]          # Processing time
}

# Draw time series
fig = timeseries_visualizer.draw(figure=fig, axes=ax_mapping)
fig = trace_visualizer.draw(figure=fig, axes=ax_mapping)

# Get axes for annotations
ax1, ax2, ax3 = axes[0], axes[1], axes[2]

# ========== Drawing components and legens ==========
# Change backend of components to PLOTLY
for component in detection_components + alg_states_components:
    component.backend = 'matplotlib'

# Draw fills and lines
for component in detection_components:
    component.draw(fig, ax1, add_legend = True)
ax1.legend(loc="upper left", fontsize=9)

for component in alg_states_components:
    component.draw(fig, ax1, add_legend = False)
    component.draw(fig, ax3, add_legend = False)
    component.draw(fig, ax2, add_legend = True)

for ax in axes:
    ax.legend(loc="upper left", fontsize=9)


#### Example: Using Timeseries and Trace Visualisers (Plotly)

In [ ]:
# Create visual components with custom styles matching the manual example
backend = DrawBackend.PLOTLY

# Create time series visualizer
timeseries_visualizer_go = UnivariateTimeseriesVisualizer(backend=backend)
timeseries_visualizer_go.set_data_provider(data)

(timeseries_visualizer_go
  .set_plot_opts(
    title="Time Series with Change Points",
    xlabel="Time Index",
    ylabel="Value",
    grid=True,
    legend=True
  ).set_draw_opts(
    color="black",
    linewidth=1.5,
    alpha=0.7
  )
)

# Create trace visualizer
trace_visualizer_go = OnlineTraceVisualizer(
    backend=backend,
    state_visualiser=DummyStateVisualizer[ShewhartControlChartState](backend=backend),
)
trace_visualizer_go.set_trace(trace)

# Configure detection function plot
(trace_visualizer_go
    .set_detection_func_plot_opts(
        title="Detection Function",
        xlabel="Time Index",
        ylabel="Detection Statistic",
        grid=True,
        legend=True
    )
    .set_detection_func_draw_opts(
        color="blue",
        linewidth=1
    )
    .set_threshold_draw_opts(
        color="red",
        linestyle="dash",
        linewidth=2,
        alpha=0.8
    )
    .set_processing_time_plot_opts(
        title="Processing Time per Step",
        xlabel="Time Index",
        ylabel="Time (seconds)",
        grid=True,
        legend=True
    )
    .set_processing_time_draw_opts(
        color="purple",
        linewidth=1,
        fill_alpha=0.3
    )
)

# Create figure with 3 subplots
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Time Series with Change Points",
        "Detection Function",
        "Processing Time per Step"
    )
)

# Map subplots to visualizers
ax_mapping = {
    "timeseries": (1, 1),                   # Time series
    "detection_function": (2, 1),     # Detection function
    "processing_time": (3,1)          # Processing time
}

# Draw time series
fig = timeseries_visualizer_go.draw(figure=fig, axes=ax_mapping)
fig = trace_visualizer_go.draw(figure=fig, axes=ax_mapping)

# Get axes for annotations
ax1, ax2, ax3 = (1,1), (2,1), (3,1)

# ========== Drawing components and legens ==========
# Draw fills and lines
for component in detection_components_go:
    component.draw(fig, ax1, add_legend = True)

for component in alg_states_components_go:
    component.draw(fig, ax1, add_legend = False)
    component.draw(fig, ax3, add_legend = False)
    component.draw(fig, ax2, add_legend = True)

# ========== Configure layout ==========
fig.update_layout(
    title="Visualization of Online Change-Point Detection using visual components",
    title_font_size=16,
    height=900,
    showlegend=True,
    hovermode="closest",
    legend={
        "x": 1.02,  # Position to the right of the plot
        "y": 0.95,     # Top-aligned
        "xanchor": "left",  # Anchor point at left of legend box
        "yanchor": "top",   # Anchor point at top of legend box
        "bgcolor": "rgba(255, 255, 255, 0.8)",  # Semi-transparent background
        "bordercolor": "black",
        "borderwidth": 1
    }
)

fig.show()

#### Example: Custom State Visualiser

In [ ]:
# Create visual components with custom styles matching the manual example
backend = DrawBackend.MATPLOTLIB

# Create time series visualizer
timeseries_visualizer = UnivariateTimeseriesVisualizer(backend=backend)
timeseries_visualizer.set_data_provider(data)

(timeseries_visualizer
    .set_plot_opts(
        xlabel="Time Index",
        ylabel="Value",
        grid=True,
    )
    .set_draw_opts(
        color="black",
        linewidth=1.5,
        alpha=0.7,
        label="Time Series"
    )
)

# Create Shewhart state visualizer
state_visualizer = ShewhartStateVisualizer(backend=backend)
state_visualizer.set_states(trace.algorithm_states)

(state_visualizer
    .set_plot_opts(
        xlabel="Time Index",
        ylabel="Value",
        grid=True,
    )
    .set_draw_opts(
        mean_color="blue",
        mean_linewidth=1.5,
        mean_label="Running Mean (μ)",
        control_limit_color="red",
        control_limit_linestyle="--",
        control_limit_linewidth=1,
        control_limit_label="Control Limit (μ ± 3σ)",
        window_mean_color="green",
        window_mean_linewidth=1,
        window_mean_label="Window Mean (x̄_w)",
        fill_alpha=0.2,
    )
)

# Create trace visualizer
trace_visualizer = OnlineTraceVisualizer(
    backend=backend,
    state_visualiser=state_visualizer,
)
trace_visualizer.set_trace(trace)

# Configure detection function plot
(trace_visualizer
    .set_detection_func_plot_opts(
        xlabel="Time Index",
        ylabel="Detection Statistic",
        grid=True,
    )
    .set_detection_func_draw_opts(
        color="blue",
        linewidth=1,
        label="Detection Function"
    )
    .set_threshold_draw_opts(
        color="red",
        linestyle="--",
        linewidth=2,
        alpha=0.8,
        label=f"Threshold = {trace.threshold:.4f}"
    )
    .set_processing_time_plot_opts(
        xlabel="Time Index",
        ylabel="Time (seconds)",
        grid=True,
    )
    .set_processing_time_draw_opts(
        color="purple",
        linewidth=1,
        fill_alpha=0.3,
        label="Processing Time"
    )
)

# Create figure with 2x2 grid layout
fig, axes = plt.subplots(2, 2, figsize=(20, 10), sharex='col')
fig.suptitle("Online Change-Point Detection with Shewhart Control Chart", fontsize=16, fontweight='bold')

# Map subplots to visualizers
ax_mapping = {
    "timeseries": axes[0, 0],                     # Time series (upper left)
    "detection_function": axes[0, 1],       # Detection function (upper right)
    "shewhart_state": axes[1, 1],           # Shewhart state (lower right)
    "processing_time": axes[1, 0],          # Processing time (lower left)
}

# Draw all visualizers
fig = timeseries_visualizer.draw(figure=fig, axes=ax_mapping)
fig = trace_visualizer.draw(figure=fig, axes=ax_mapping)

# Get axes for annotations
ax_ts = axes[0, 0]      # Time series
ax_det = axes[0, 1]     # Detection function
ax_pt = axes[1, 0]      # Processing time
ax_state = axes[1, 1]   # Shewhart state

# ========== Drawing components and legens ==========
# Draw fills and lines
for component in detection_components:
    component.draw(fig, ax_ts, add_legend = True)
  
for component in alg_states_components:
    component.draw(fig, ax_ts, add_legend = False)
    component.draw(fig, ax_det, add_legend = False)
    component.draw(fig, ax_state, add_legend = False)
    component.draw(fig, ax_pt, add_legend = True)

for ax in [ax_ts, ax_det, ax_pt, ax_state]:
    ax.legend(loc="best", fontsize=9)

# Adjust layout
plt.tight_layout()
plt.show()

### Level III: Presets. Univariate Online Change Point Visualiser

The top level component that ensures consistent style thorugh various visualisers and components

In [ ]:
"""
Simplified example using OnlineCpdPlotter with all defaults.
"""

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from pysatl_cpd.analysis.visualization.typedefs import DrawBackend
from pysatl_cpd.analysis.visualization.online_cpd_plotter import OnlineCpdPlotter
from pysatl_cpd.core.data_providers import NDArrayUnivariateProvider
from pysatl_cpd.core.online import OnlineCpdSolver, OnlineDetectionTrace
from pysatl_cpd.algorithms.online.shewhart_control_chart import ShewhartControlChart


def generate_labeled_data(
    means: list[float],
    lengths: list[int],
    var: float = 1.0,
) -> NDArrayUnivariateProvider:
    """Generate synthetic time series data with a change point."""
    raw_data = np.empty(0)
    for mean, length in zip(means, lengths):
        segment = norm.rvs(size=length, loc=mean, scale=np.sqrt(var))
        raw_data = np.concatenate((raw_data, segment))
    return NDArrayUnivariateProvider(raw_data)

# Generate data with change point at index 1000
data_provider = generate_labeled_data([0, 1], [1000, 1000])

# Run detection
algorithm = ShewhartControlChart(learning_period_size=100, window_size=50)
solver = OnlineCpdSolver(data_provider, algorithm, threshold=2.5, skip_period=50)
steps = list(solver.run())
trace = OnlineDetectionTrace.from_online_detection_steps(threshold=2.5, steps=steps)

# Create plotter with defaults
plotter = OnlineCpdPlotter(
    backend=DrawBackend.MATPLOTLIB,
    data_provider=data_provider,
    detection_trace=trace,
)

# Set ground truth data (only domain-specific info needed)
plotter.set_ground_truth([1000], margin=10)

# Create figure and draw
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Online Change-Point Detection with Shewhart Control Chart", fontsize=16)

ax_mapping = {
    "timeseries": axes[0],
    "detection_function": axes[1],
    "processing_time": axes[2],
}

fig = plotter.draw(figure=fig, axes=ax_mapping)
plt.tight_layout()
plt.show()


## Acme. Benchmarking and Metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pysatl_cpd.analysis.visualization.benchmarking import (
    BenchmarkPlotter,
    PrAucVisualizer,
    ThresholdBasedMetricVisualizer,
)

# Synthetic benchmark table for visualization demo
rng = np.random.default_rng(42)
thresholds = np.linspace(0.1, 1.0, 20)

recall = np.clip(1.0 - 0.75 * thresholds + rng.normal(0, 0.02, size=thresholds.size), 0.0, 1.0)
precision = np.clip(0.3 + 0.65 * thresholds + rng.normal(0, 0.02, size=thresholds.size), 0.0, 1.0)
f1 = (2 * precision * recall) / np.clip(precision + recall, 1e-12, None)

benchmark_df = pd.DataFrame(
    {
        "threshold": thresholds,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_delay": 55 - 35 * thresholds + rng.normal(0, 1.0, size=thresholds.size),
        "median_delay": 45 - 30 * thresholds + rng.normal(0, 1.0, size=thresholds.size),
        "arl": 1.0 / thresholds,
    }
)

# Legacy order: [F1, Precision/Recall, PR-AUC, Delays]
plotter = (
    BenchmarkPlotter()
    .set_benchmark_table(benchmark_df)
    .set_metrics(
        {
            "f1": ThresholdBasedMetricVisualizer(
                y_metrics=["f1"],
                title="F1",
                ylabel="F1 value",
            ),
            "precision_recall": ThresholdBasedMetricVisualizer(
                y_metrics=["precision", "recall"],
                title="Precision and Recall",
                ylabel="Precision/Recall value",
            ),
            "pr_auc": PrAucVisualizer(label="PR-AUC"),
            "delays": ThresholdBasedMetricVisualizer(
                y_metrics=["mean_delay", "median_delay"],
                title="Detection delay",
                ylabel="Delay (in sample points)",
            ),
        }
    )
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
ax_mapping = {
    "f1": axes[0, 0],
    "precision_recall": axes[0, 1],
    "pr_auc": axes[1, 0],
    "delays": axes[1, 1],
}

plotter.draw(figure=fig, axes=ax_mapping)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from plotly.subplots import make_subplots

from pysatl_cpd.analysis.visualization.benchmarking import (
    BenchmarkPlotter,
    PrAucVisualizer,
    ThresholdBasedMetricVisualizer,
)

# Synthetic benchmark table for Plotly demo
rng = np.random.default_rng(42)
thresholds = np.linspace(0.1, 1.0, 20)

recall = np.clip(1.0 - 0.75 * thresholds + rng.normal(0, 0.02, size=thresholds.size), 0.0, 1.0)
precision = np.clip(0.3 + 0.65 * thresholds + rng.normal(0, 0.02, size=thresholds.size), 0.0, 1.0)
f1 = (2 * precision * recall) / np.clip(precision + recall, 1e-12, None)

benchmark_df_plotly = pd.DataFrame(
    {
        "threshold": thresholds,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_delay": 55 - 35 * thresholds + rng.normal(0, 1.0, size=thresholds.size),
        "median_delay": 45 - 30 * thresholds + rng.normal(0, 1.0, size=thresholds.size),
        "arl": 1.0 / thresholds,
    }
)

# Legacy order: [F1, Precision/Recall, PR-AUC, Delays]
plotter_plotly = (
    BenchmarkPlotter()
    .set_benchmark_table(benchmark_df_plotly)
    .set_metrics(
        {
            "f1": ThresholdBasedMetricVisualizer(
                y_metrics=["f1"],
                title="F1",
                ylabel="F1 value",
            ),
            "precision_recall": ThresholdBasedMetricVisualizer(
                y_metrics=["precision", "recall"],
                title="Precision and Recall",
                ylabel="Precision/Recall value",
            ),
            "pr_auc": PrAucVisualizer(label="PR-AUC"),
            "delays": ThresholdBasedMetricVisualizer(
                y_metrics=["mean_delay", "median_delay"],
                title="Detection delay",
                ylabel="Delay (in sample points)",
            ),
        }
    )
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "F1",
        "Precision and Recall",
        "PR-AUC",
        "Detection delay",
    ),
)

ax_mapping = {
    "f1": (1, 1),
    "precision_recall": (1, 2),
    "pr_auc": (2, 1),
    "delays": (2, 2),
}

fig = plotter_plotly.draw(figure=fig, axes=ax_mapping)
fig.update_layout(height=800, width=1200, title_text="Benchmark metrics (Plotly, legacy order)")
fig.show()